## 1. Environment

In [ ]:
!pip -q install -U "transformers>=4.44" accelerate sentencepiece

import torch

HAS_CUDA = torch.cuda.is_available()
CAP = torch.cuda.get_device_capability()[0] if HAS_CUDA else 0
DTYPE = torch.bfloat16 if CAP >= 8 else torch.float16

if HAS_CUDA:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'gpu {name} | sm_{CAP}0 | {vram:.1f} GB | dtype {DTYPE}')
else:
    print('No GPU - Runtime > Change runtime type > T4 GPU.')

gpu Tesla T4 | sm_70 | 15.6 GB | dtype torch.float16


In [ ]:
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

## 2. Hugging Face login (gated rungs)

In [ ]:
from huggingface_hub import login

login()

## 3. Upload the headline prompts

In [ ]:
import pandas as pd
from google.colab import files

up = files.upload()
prompts = pd.read_csv(next(iter(up)), dtype=str).fillna('')
assert {'prompt_id', 'outlet', 'period', 'section', 'headline'} <= set(prompts.columns)
print(len(prompts), 'prompts;', prompts.groupby('outlet').size().to_dict())
prompts.head()

Saving gen_prompts.csv to gen_prompts (1).csv
600 prompts; {'gript': 150, 'irish_examiner': 150, 'rte': 150, 'the_liberal': 150}


,prompt_id,outlet,period,section,headline
0,p0000,gript,post,Comment Ireland|Featured,Dispute over Irish language signage exposes fl...
1,p0001,gript,post,Irish News,Fundraiser for “gentle” girl who died in River...
2,p0002,gript,post,Irish News,Public meeting in Meath to hear there has been...
3,p0003,gript,post,World News,Study: 95% of young women on testosterone for ...
4,p0004,gript,post,World News,Musk says Twitter interfered in elections


## 4. The model ladder — T4

In [ ]:
# short name : (hf_id, era_year, kind)
MODELS = {
    'gpt2-large':  ('openai-community/gpt2-large',        2019, 'base'),
    'gptneo-1.3b': ('EleutherAI/gpt-neo-1.3B',            2021, 'base'),
    'llama2-7b':   ('meta-llama/Llama-2-7b-chat-hf',      2023, 'chat'),
    'mistral-7b':  ('mistralai/Mistral-7B-Instruct-v0.1', 2023, 'chat'),
    'qwen2.5-7b':  ('Qwen/Qwen2.5-7B-Instruct',           2024, 'chat'),
}

PROMPT_VERSION = 'irish-register-v3'   # keep in sync with generate_frontier_api.py
PER_OUTLET     = 150                   # articles per outlet per model
BATCH_SIZE     = 2
MAX_NEW_TOK    = 700
TEMPERATURE    = 0.9
TOP_P          = 0.95
MIN_WORDS      = 100                   # drop truncated / refused / garbage generations
SEED           = 37
OUT_CSV        = 'generated_irish_ai.csv'

import torch

torch.manual_seed(SEED)
work = (prompts.groupby('outlet', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), PER_OUTLET), random_state=SEED)))
print(f'{len(work)} prompts x {len(MODELS)} models = {len(work) * len(MODELS)} generations')
print(work.groupby('outlet').size().to_dict())

600 prompts x 5 models = 3000 generations
{'gript': 150, 'irish_examiner': 150, 'rte': 150, 'the_liberal': 150}


/tmp/ipykernel_716/3467609381.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), PER_OUTLET), random_state=SEED)))


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
import os

SAVE_DIR = '/content/drive/MyDrive/eire_generation'
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_CSV = f'{SAVE_DIR}/generated_irish_ai.csv'
print('writing to', OUT_CSV, '| already exists:', os.path.exists(OUT_CSV))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
writing to /content/drive/MyDrive/eire_generation/generated_irish_ai.csv | already exists: True


In [ ]:
import os
import shutil

LOCAL_CSV = '/content/generated_irish_ai.csv'
DRIVE_CSV = '/content/drive/MyDrive/eire_generation/generated_irish_ai.csv'
os.makedirs(os.path.dirname(DRIVE_CSV), exist_ok=True)
if os.path.exists(DRIVE_CSV) and not os.path.exists(LOCAL_CSV):
    shutil.copy(DRIVE_CSV, LOCAL_CSV)          # fresh runtime: seed local from Drive
OUT_CSV = LOCAL_CSV                            # writes go to fast local disk
SYNC_EVERY = 300                              # seconds between Drive checkpoints

def sync_to_drive():
    try:
        shutil.copy(LOCAL_CSV, DRIVE_CSV)
        print('  synced to Drive', flush=True)
    except OSError as e:
        print('  Drive sync failed, remounting:', e, flush=True)
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=True)
            shutil.copy(LOCAL_CSV, DRIVE_CSV)
            print('  synced after remount', flush=True)
        except OSError as e2:
            print('  still failing, will retry next interval:', e2, flush=True)

## 5. The prompt (v3) — register floor + real outlet

In [ ]:
SYSTEM_PROMPT = (
    'You are writing a news article in English for an Irish national news outlet, in the '
    'style of Irish print register and online news reporting.\n'
    '- Use Irish English spelling and usage (organisation, realise, '
    'defence, programme, travelled), euro for money, and metric units.\n'
    '- Where relevant, name Irish institutions and roles as Irish reporting does, for example: '
    'the Dáil, the Oireachtas, a TD, the Taoiseach, the Tánaiste, the Gardaí, the '
    'HSE, government departments, the District, Circuit and High Courts.\n'
    'Write only the article body: no headline, byline, dateline, sign-off, '
    'section headings, lists, or markdown.'
)

# Real outlet + domain.the model supplies house style.
OUTLET_REF = {
    'rte':            'RTÉ News (rte.ie)',
    'irish_examiner': 'the Irish Examiner (irishexaminer.com)',
    'the_liberal':    'The Liberal (theliberal.ie)',
    'gript':          'Gript (gript.ie)',
}

print(SYSTEM_PROMPT)

You are writing a news article in English for an Irish national news outlet, in the style of Irish print register and online news reporting.
- Use Irish English spelling and usage (organisation, realise, defence, programme, travelled), euro for money, and metric units.
- Where relevant, name Irish institutions and roles as Irish reporting does, for example: the Dáil, the Oireachtas, a TD, the Taoiseach, the Tánaiste, the Gardaí, the HSE, government departments, the District, Circuit and High Courts.
Write only the article body: no headline, byline, dateline, sign-off, section headings, lists, or markdown.


## 6. Helpers — render, clean, filter

In [ ]:
import re

_REFUSAL = re.compile(r"\b(i can(?:no|')t|i'm sorry|as an ai|i am unable|"
                      r"i cannot (?:access|provide))\b", re.I)
# reasoning markers differ by family; strip them before length/quality checks
_THINK = [re.compile(r'<think>.*?</think>', re.S),
          re.compile(r'<\|?thought\|?>.*?<\|?/?thought\|?>', re.S)]


def render(row, kind, tok):
    """Build the model input for one prompt row."""
    ref = OUTLET_REF.get(row['outlet'], 'an Irish national news outlet')
    if kind == 'base':
        # base models cannot follow instructions - a completion header is the fair low end
        return (f'The following is a news article from {ref}.\n\n'
                f"Headline: {row['headline']}\n\nArticle:\n")
    section = f"{row['section']} " if row['section'] else ''
    user = (f"Write a {section}article for {ref}, in that outlet's own style, "
            f"under this headline:\n\n{row['headline']}")
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': user}]
    for kwargs in ({'enable_thinking': False}, {}):
        try:
            return tok.apply_chat_template(messages, tokenize=False,
                                           add_generation_prompt=True, **kwargs)
        except (TypeError, ValueError):
            continue
    # some older templates reject a system role - fold it into the user turn
    merged = [{'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + user}]
    return tok.apply_chat_template(merged, tokenize=False, add_generation_prompt=True)


def clean(text, headline):
    """Strip reasoning blocks, echoed headlines and stray markdown."""
    out = text
    for pat in _THINK:
        out = pat.sub('', out)
    out = out.strip().lstrip('#* ').strip()
    if out.lower().startswith(headline.lower()[:40]):
        out = out[len(headline):].lstrip(' :\n-').strip()
    return out


def usable(text):
    """Long enough and not a refusal."""
    return len(text.split()) >= MIN_WORDS and not _REFUSAL.search(text[:200])